# Libary for pdf generation

In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.0 MB/s eta 0:00:00


# To handle arabic language

In [ ]:
pip install arabic-reshaper python-bidi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 6.7 MB/s eta 0:00:00


In [ ]:
import arabic_reshaper
from bidi.algorithm import get_display
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

In [ ]:
pdfmetrics.registerFont(TTFont('Arabic', '/content/Amiri-Regular.ttf'))
def fix_arabic_text(text):
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)
    return bidi_text

In [ ]:
def clean_text(text):
    # remove duplicated serialized block if exists
    text = re.sub(r"\('.*", "", text, flags=re.DOTALL)
    return text

In [ ]:
from google.colab import userdata

In [ ]:
import re
import os
import gradio as gr
from huggingface_hub import InferenceClient
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Image as RLImage, Paragraph, Spacer
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from PIL import Image

# -------------------------------
# Hugging Face Clients
# -------------------------------
HF_TOKEN = userdata.get('api-huggingFace')

text_client = InferenceClient(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    token=HF_TOKEN
)
# -------------------------------
# Parse Output
# -------------------------------
def parse_story(text):
    pattern = r"\*{0,2}Page\s*(\d+)\*{0,2}.*?English:\s*(.*?)\nArabic:\s*(.*?)\nImage Prompt:\s*(.*?)(?=\n\*{0,2}Page|\Z)"

    matches = re.findall(pattern, text, re.IGNORECASE | re.DOTALL)

    pages = []
    for m in matches:
        pages.append({
            "page": m[0],
            "english": m[1].strip(),
            "arabic": m[2].strip(),
            "prompt": m[3].strip()
        })

    return pages

# -------------------------------
# Generate Story
# -------------------------------
def generate_story(theme):

    prompt = f"""
    Create a 5-page bilingual storybook for an 8-year-old Algerian child.

    Requirements:
    - Each page contains:
      1. Simple English sentence
      2. Arabic translation
    - Use easy vocabulary
    - Include a moral lesson
    - Theme: {theme}

    STRICT FORMAT:

    **Page 1**
    English: ...
    Arabic: ...
    Image Prompt: ...

    **Page 2**
    ...

    Do NOT add any extra explanation before or after.
    """

    response = text_client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800,
        temperature=0.7,
    )
    raw_text = response.choices[0].message["content"]
    raw_text = clean_text(raw_text)

    pages = parse_story(raw_text)

    print("pages: ",pages)
    return raw_text, pages

In [ ]:
generate_story("friendship and desert adventure")

pages:  [{'page': '1', 'english': 'In the vast Sahara desert, there lived two friends named Ahmed and Omar.', 'arabic': 'في الصحراء الكبرى، كانت تعيش صديقان هما أحمد وعمر.', 'prompt': 'A colorful illustration of Ahmed and Omar standing in front of a vast desert landscape.'}, {'page': '2', 'english': 'They loved to explore the dunes and play in the sand.', 'arabic': 'يحبونBoth أن يفتحوا على جبال الرمال ويمضوا في الرمال.', 'prompt': 'An illustration of Ahmed and Omar running and playing in the sand.'}, {'page': '3', 'english': 'One day, they came across a lost camel named Samira.', 'arabic': 'ذات يوم، وجدوا دودة بحمولة مفقودة اسمها ساميرة.', 'prompt': 'An illustration of Ahmed and Omar helping Samira, who is looking worried.'}, {'page': '4', 'english': 'Ahmed and Omar helped Samira find her way back home. She was very grateful.', 'arabic': 'ساعدت SamuelBoth أحمد وعمر ساميرة على العودة إلى بيها. كانت شاكرة للغاية.', 'prompt': 'An illustration of Ahmed, Omar, and Samira walking together, w

('**Page 1**\nEnglish: In the vast Sahara desert, there lived two friends named Ahmed and Omar.\nArabic: في الصحراء الكبرى، كانت تعيش صديقان هما أحمد وعمر.\nImage Prompt: A colorful illustration of Ahmed and Omar standing in front of a vast desert landscape.\n\n**Page 2**\nEnglish: They loved to explore the dunes and play in the sand.\nArabic: يحبونBoth أن يفتحوا على جبال الرمال ويمضوا في الرمال.\nImage Prompt: An illustration of Ahmed and Omar running and playing in the sand.\n\n**Page 3**\nEnglish: One day, they came across a lost camel named Samira.\nArabic: ذات يوم، وجدوا دودة بحمولة مفقودة اسمها ساميرة.\nImage Prompt: An illustration of Ahmed and Omar helping Samira, who is looking worried.\n\n**Page 4**\nEnglish: Ahmed and Omar helped Samira find her way back home. She was very grateful.\nArabic: ساعدت SamuelBoth أحمد وعمر ساميرة على العودة إلى بيها. كانت شاكرة للغاية.\nImage Prompt: An illustration of Ahmed, Omar, and Samira walking together, with a desert landscape in the backg

In [ ]:
# -------------------------------
# Generate Images
# -------------------------------
def generate_images(pages):
    image_paths = []
    image_client = InferenceClient(
    #provider="fal-ai",
    provider="auto",
    api_key=HF_TOKEN)

    for i, page in enumerate(pages):
        img = image_client.text_to_image(
            page["prompt"],
            model="stabilityai/stable-diffusion-xl-base-1.0"
        )

        path = f"page_{i}.png"
        img.save(path)
        image_paths.append(path)

    return image_paths


In [ ]:
from reportlab.lib.styles import ParagraphStyle

def create_pdf(pages, image_paths):
    doc = SimpleDocTemplate("storybook.pdf")
    elements = []

    # Custom styles
    arabic_style = ParagraphStyle(
        name='ArabicStyle',
        fontName='Arabic',
        fontSize=12,
        alignment=2,  # RIGHT ALIGN
    )

    english_style = ParagraphStyle(
        name='EnglishStyle',
        fontName='Helvetica',
        fontSize=12,
        alignment=0,  # LEFT ALIGN
    )

    for i, page in enumerate(pages):

        arabic_fixed = fix_arabic_text(page['arabic'])

        english_para = Paragraph(
            f"<b>English:</b> {page['english']}",
            english_style
        )

        arabic_para = Paragraph(
            f"<b>Arabic:</b> {arabic_fixed}",
            arabic_style
        )

        img = RLImage(image_paths[i], width=200, height=200)

        # Row 1: Text | Image
        row1 = [[english_para, img]]

        # Row 2: Image | Arabic
        row2 = [[img, arabic_para]]

        table = Table(row1 + row2)
        table.setStyle(TableStyle([
            ('GRID', (0,0), (-1,-1), 1, colors.black),
        ]))

        elements.append(table)
        elements.append(Spacer(1, 20))

    doc.build(elements)

    return "storybook.pdf"

In [ ]:
# -------------------------------
# Main Pipeline
# -------------------------------
def run_app(theme):
    raw_text, pages = generate_story(theme)
    print(raw_text)

    filtered_output = ""
    for p in pages:
        filtered_output += f"""
Page {p['page']}
English: {p['english']}
Arabic: {p['arabic']}
Image Prompt: {p['prompt']}

"""

    image_paths = generate_images(pages)
    pdf_path = create_pdf(pages, image_paths)

    return filtered_output, pdf_path

In [ ]:

# -------------------------------
# Gradio UI
# -------------------------------
themes = [
    "friendship and desert adventure",
    "animals in the forest",
    "space adventure",
    "school life",
    "kindness and helping others"
]

with gr.Blocks() as demo:
    gr.Markdown("# 📚 AI Storybook Generator")

    gr.HTML("<h5>Through this user interface, you can generate a PDF storybook for an 8-year-old Algerian child. The short story will be in English and Arabic languages, illustrated by images.</h5>")
    theme_dropdown = gr.Dropdown(choices=themes, label="Select Theme")

    btn = gr.Button("Generate Storybook")

    output_text = gr.Textbox(label="Filtered Story Output", lines=20)
    pdf_file = gr.File(label="Download PDF")

    btn.click(
        fn=run_app,
        inputs=theme_dropdown,
        outputs=[output_text, pdf_file]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7b10de13896ea76083.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
